# Week 3 SQL Assignment

# Window Functions

## Objective

The objective of this notebook is to understand and implement SQL Window Functions using the Superstore dataset.

Window Functions help perform calculations across rows without grouping the data into a single result.

In this notebook, ROW_NUMBER() and RANK() functions are used to solve business-related problems.

## Step 1 : Import Required Libraries

In this step, we import the required Python libraries.

- pandas is used to display SQL query results.
- sqlite3 is used to connect with the SQLite database.

In [1]:
# Import pandas library for displaying query results
import pandas as pd

# Import sqlite3 library for database operations
import sqlite3

## Step 2 : Connect to the Database

A connection is established with the SQLite database created in the previous notebook.

This connection will be used to execute SQL queries.

In [2]:
# Connect to SQLite database

conn = sqlite3.connect("../superstore.db")

# Create cursor object

cursor = conn.cursor()

## Step 3 : Verify Available Tables

Before executing window function queries, verify that the required tables are available in the database.

In [3]:
# Display available tables

pd.read_sql("""

SELECT name

FROM sqlite_master

WHERE type='table';

""", conn)

,name
0,customers
1,products
2,superstore_raw
3,orders


### Observation

The required tables are available in the SQLite database.

This confirms that the database setup has been completed successfully and is ready for implementing Window Functions.

## Introduction to Window Functions

Window Functions perform calculations across a set of rows while keeping each row separate.

Unlike GROUP BY, Window Functions do not combine multiple rows into one result.

They are useful for ranking, numbering and analytical calculations.

## Real-World Example

Suppose a company wants to rank its customers based on total sales without removing individual customer records.

Using Window Functions such as RANK() and ROW_NUMBER(), SQL can perform ranking and numbering operations while preserving every row of the dataset.

This makes analytical reporting simpler and more efficient.

## Why do we use Window Functions?

Window Functions simplify analytical queries.

They allow ranking, numbering and comparison of rows without losing individual row details.

They are widely used in reporting and business analysis.

## Query 5

### Problem Statement

Rank all customers based on total sales. (Window Function)  

### Understanding the Problem

The total sales of each customer are calculated first.

The RANK() function is then used to assign rankings based on total sales.

Customers with higher sales receive better ranks.

In [4]:
# Query 5
# Rank customers based on total sales

query5 = """
WITH customer_sales AS
(
SELECT
customer_id,
SUM(sales) AS total_sales
FROM orders
GROUP BY customer_id
)
SELECT
customer_id,
total_sales,
RANK()
OVER(
ORDER BY total_sales DESC
)
AS customer_rank
FROM customer_sales;
"""
pd.read_sql(query5, conn)

,customer_id,total_sales,customer_rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3
3,TA-21385,14595.620,4
4,AB-10105,14473.571,5
...,...,...,...
788,RS-19870,22.328,789
789,MG-18205,16.739,790
790,CJ-11875,16.520,791
791,LD-16855,5.304,792


### Observation

The query ranks customers based on their cumulative sales.

Customers with higher total sales receive better ranks, making it easier to identify the most valuable customers.

## Query 6

### Problem Statement

Assign row numbers to each order within a customer.

### Understanding the Problem

Each customer may have placed multiple orders.

The ROW_NUMBER() function assigns a sequential number to every order of a customer.

The numbering restarts for every new customer because PARTITION BY is used.

In [5]:
# Query 6
# Assign row numbers to each order within a customer.

query6 = """

SELECT 

customer_id,
order_id,
sales,
ROW_NUMBER()
OVER(
PARTITION BY customer_id
ORDER BY sales DESC
)
AS row_number
FROM orders;
"""

pd.read_sql(query6, conn)

,customer_id,order_id,sales,row_number
0,AA-10315,CA-2016-103982,3930.072,1
1,AA-10315,CA-2014-128055,673.568,2
2,AA-10315,CA-2016-103982,431.976,3
3,AA-10315,CA-2017-147039,362.940,4
4,AA-10315,CA-2014-128055,52.980,5
...,...,...,...,...
9988,ZD-21925,CA-2017-141481,61.440,5
9989,ZD-21925,CA-2014-143336,22.720,6
9990,ZD-21925,US-2016-147991,16.720,7
9991,ZD-21925,CA-2016-152471,15.984,8


### Observation

The ROW_NUMBER() function assigns a sequential number to every order within each customer group.

The numbering restarts from one whenever a new customer is encountered due to the PARTITION BY clause.

## Query 7

### Problem Statement
Display top 3 customers based on total sales.

### Understanding the Problem

After ranking customers based on total sales, only the top three customers are selected.

This helps identify the highest revenue-generating customers.

In [6]:
# Query 7
# Display top 3 customers based on total sales

query7 = """

WITH customer_sales AS

(
SELECT
customer_id,
SUM(sales) AS total_sales
FROM orders
GROUP BY customer_id
)

SELECT *
FROM(
SELECT
customer_id,
total_sales,
RANK()
OVER(
    ORDER BY total_sales DESC
)
AS customer_rank
FROM customer_sales
)
WHERE customer_rank <=3;
"""

pd.read_sql(query7, conn)


,customer_id,total_sales,customer_rank
0,SM-20320,25043.050,1
1,TC-20980,19052.218,2
2,RB-19360,15117.339,3


### Observation

The query identifies the three customers with the highest cumulative sales.

These customers contribute significantly to the overall revenue generated by the business.

## Key Learning

While working on this notebook, I learned how Window Functions perform calculations across rows without grouping the data.

Understanding ROW_NUMBER() and RANK() improved my ability to solve ranking and analytical business problems using SQL.

## Conclusion

This notebook demonstrated the practical use of Window Functions in SQL.

ROW_NUMBER() was used to assign row numbers within customer groups.

RANK() was used to rank customers based on total sales.

The top three customers were identified using ranking, from the Superstore dataset.

## Reflection

Working on Window Functions helped me understand how SQL can perform advanced analytical operations while preserving row-level details.

Implementing ROW_NUMBER() and RANK() improved my understanding of customer ranking and order analysis in real-world business scenarios.

In [7]:
# Close the database connection after completing the analysis

conn.close()